
# Домашнє завдання №1  
## ReAct-агент у LangGraph для домашньої енергосистеми

**Студент:** Антон Бабенко  
**Формат:** Google Colab / Jupyter Notebook  
**LLM-провайдер:** Google Gemini через LangChain  
**Домен:** домашня енергетика  

---

## Як користуватися цим ноутбуком

Позначення:

- 🟩 **ОБОВ’ЯЗКОВО** — комірку потрібно запустити.
- 🟨 **ДЕМОНСТРАЦІЯ** — запустити один раз, щоб у ноутбуці залишився результат для викладача.
- 🟥 **НЕ ЗАПУСКАТИ ПОВТОРНО** — комірка витрачає Gemini API-квоту.
- ⬜ **ОПЦІЙНО** — можна не запускати.

### Важливо про API-виклики

Локальні тести Pydantic і tools **не витрачають API-квоту**.  
Gemini використовується лише після секції підключення LLM.

Для економії квоти:

1. Не запускайте повторно окремі тести вибору tools.
2. Не використовуйте окремий LLM-formatter.
3. Фінальний structured output формується локально через Pydantic.
4. Фінальні 4 тест-кейси запускаються лише один раз.



# 0. План роботи та відповідність вимогам

| Вимога | Де реалізовано |
|---|---|
| 3–4 tools з Pydantic v2 | Розділ 1 |
| `Field(description=...)` | У кожній Pydantic-схемі |
| `field_validator` | У кожній Pydantic-схемі |
| Docstring для LLM | У кожному tool |
| Unit tests / демонстрація | Розділ 1.6 |
| LangGraph `StateGraph` | Розділ 3 |
| Вузол `agent` | Розділ 3.3 |
| Вузол `tools` через `ToolNode` | Розділ 3.4 |
| Conditional edge / router | Розділ 3.5 |
| Structured output | Розділ 4 |
| `max_steps` | Розділ 5 |
| `timeout` | Розділ 5 |
| Loop detection | Розділ 5 |
| `trajectory.json` | Розділ 6 |
| 3–5 тест-кейсів | Розділ 7 |
| `test_results.json` | Розділ 7 |
| `README.md` | Розділ 8 |

Умова дозволяє оформити рішення як один Jupyter Notebook / Google Colab.  
Окремо генеруються файли `trajectory.json`, `test_results.json` та `README.md`.


## 0.1 🟩 ОБОВ’ЯЗКОВО — встановлення бібліотек

In [1]:
!pip install -q -U langgraph langchain langchain-core langchain-google-genai "pydantic>=2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 4.0 MB/s eta 0:00:00


## 0.2 🟩 ОБОВ’ЯЗКОВО — імпорти та перевірка версій

In [2]:
import os
import json
import time
import hashlib
import warnings
from datetime import datetime, timezone
from typing import TypedDict, Annotated, Literal
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
import operator

import pydantic
import langchain
from pydantic import BaseModel, Field, field_validator, ConfigDict

from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage,
    ToolMessage,
)
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

print("Pydantic:", pydantic.__version__)
print("LangChain:", langchain.__version__)
print("✅ Бібліотеки імпортовано")

Pydantic: 2.13.4
LangChain: 1.3.14
✅ Бібліотеки імпортовано



# 1. Завдання 1 — tools з Pydantic-схемами

Створюються чотири реалістичні інструменти:

1. `calculate_energy` — споживання електроенергії.
2. `estimate_battery_runtime` — час автономної роботи акумулятора.
3. `calculate_solar_generation` — оцінка генерації сонячних панелей.
4. `check_inverter_load` — перевірка завантаження інвертора.

Кожен tool має:

- окремий `BaseModel`;
- `Field(description=...)`;
- `field_validator`;
- докладний docstring, який читає LLM.


## 1.1 🟩 ОБОВ’ЯЗКОВО — tool `calculate_energy`

In [3]:
class EnergyConsumptionInput(BaseModel):
    """Параметри для розрахунку спожитої електроенергії."""

    model_config = ConfigDict(str_strip_whitespace=True)

    power_watts: float = Field(
        description="Потужність одного приладу у ватах, більше 0."
    )
    hours: float = Field(
        description="Тривалість роботи приладу в годинах, від 0.01 до 24."
    )
    devices_count: int = Field(
        default=1,
        description="Кількість однакових приладів, від 1 до 100."
    )

    @field_validator("power_watts")
    @classmethod
    def validate_power(cls, value: float) -> float:
        if value <= 0:
            raise ValueError("Потужність повинна бути більшою за 0 Вт")
        if value > 100_000:
            raise ValueError("Потужність не може перевищувати 100 000 Вт")
        return value

    @field_validator("hours")
    @classmethod
    def validate_hours(cls, value: float) -> float:
        if not 0.01 <= value <= 24:
            raise ValueError("Час роботи має бути від 0.01 до 24 годин")
        return value

    @field_validator("devices_count")
    @classmethod
    def validate_devices_count(cls, value: int) -> int:
        if not 1 <= value <= 100:
            raise ValueError("Кількість приладів має бути від 1 до 100")
        return value


@tool(args_schema=EnergyConsumptionInput)
def calculate_energy(
    power_watts: float,
    hours: float,
    devices_count: int = 1,
) -> str:
    """Розрахувати споживання електроенергії одним або кількома приладами.

    Використовуйте цей інструмент, коли користувач запитує, скільки
    електроенергії споживатиме прилад за певний час.

    Приклади:
    - обігрівач 2000 Вт протягом 5 годин;
    - три лампи по 10 Вт протягом 8 годин.

    Returns:
        Результат із сумарною потужністю та енергією у кВт·год.
    """
    total_power_watts = power_watts * devices_count
    energy_kwh = total_power_watts * hours / 1000

    return (
        f"Кількість приладів: {devices_count}. "
        f"Загальна потужність: {total_power_watts:.2f} Вт. "
        f"Час роботи: {hours:.2f} год. "
        f"Споживання електроенергії: {energy_kwh:.3f} кВт·год."
    )

## 1.2 🟩 ОБОВ’ЯЗКОВО — tool `estimate_battery_runtime`

In [4]:
class BatteryRuntimeInput(BaseModel):
    """Параметри для оцінки часу роботи акумулятора."""

    model_config = ConfigDict(str_strip_whitespace=True)

    battery_capacity_kwh: float = Field(
        description="Повна ємність акумулятора у кВт·год, більше 0."
    )
    current_soc_percent: float = Field(
        description="Поточний рівень заряду акумулятора, від 0 до 100%."
    )
    reserve_soc_percent: float = Field(
        default=10,
        description="Мінімальний резерв заряду, від 0 до 95%."
    )
    load_watts: float = Field(
        description="Середня потужність навантаження у ватах, більше 0."
    )
    efficiency_percent: float = Field(
        default=90,
        description="Загальний ККД акумулятора та інвертора, від 50 до 100%."
    )

    @field_validator("battery_capacity_kwh")
    @classmethod
    def validate_capacity(cls, value: float) -> float:
        if not 0 < value <= 1000:
            raise ValueError("Ємність має бути більшою за 0 і не більшою 1000 кВт·год")
        return value

    @field_validator("current_soc_percent")
    @classmethod
    def validate_current_soc(cls, value: float) -> float:
        if not 0 <= value <= 100:
            raise ValueError("Поточний SOC має бути від 0 до 100%")
        return value

    @field_validator("reserve_soc_percent")
    @classmethod
    def validate_reserve_soc(cls, value: float) -> float:
        if not 0 <= value <= 95:
            raise ValueError("Резервний SOC має бути від 0 до 95%")
        return value

    @field_validator("load_watts")
    @classmethod
    def validate_load(cls, value: float) -> float:
        if not 0 < value <= 1_000_000:
            raise ValueError("Навантаження має бути більшим за 0 Вт")
        return value

    @field_validator("efficiency_percent")
    @classmethod
    def validate_efficiency(cls, value: float) -> float:
        if not 50 <= value <= 100:
            raise ValueError("ККД має бути від 50 до 100%")
        return value


@tool(args_schema=BatteryRuntimeInput)
def estimate_battery_runtime(
    battery_capacity_kwh: float,
    current_soc_percent: float,
    load_watts: float,
    reserve_soc_percent: float = 10,
    efficiency_percent: float = 90,
) -> str:
    """Оцінити час автономної роботи навантаження від акумулятора.

    Використовуйте, коли потрібно визначити, скільки годин акумулятор
    живитиме будинок, котел, сервер або інше навантаження.

    Інструмент враховує поточний SOC, резерв та ККД системи.
    """
    if current_soc_percent <= reserve_soc_percent:
        return (
            f"Поточний заряд {current_soc_percent:.1f}% не перевищує "
            f"резерв {reserve_soc_percent:.1f}%. Доступної енергії немає."
        )

    usable_soc = current_soc_percent - reserve_soc_percent
    available_kwh = (
        battery_capacity_kwh
        * usable_soc / 100
        * efficiency_percent / 100
    )
    runtime_hours = available_kwh / (load_watts / 1000)
    hours = int(runtime_hours)
    minutes = int(round((runtime_hours - hours) * 60))

    if minutes == 60:
        hours += 1
        minutes = 0

    return (
        f"Доступна енергія: {available_kwh:.3f} кВт·год. "
        f"Навантаження: {load_watts:.2f} Вт. "
        f"Орієнтовний час роботи: {hours} год {minutes} хв."
    )

## 1.3 🟩 ОБОВ’ЯЗКОВО — tool `calculate_solar_generation`

In [5]:
class SolarGenerationInput(BaseModel):
    """Параметри для оцінки генерації сонячної електростанції."""

    model_config = ConfigDict(str_strip_whitespace=True)

    panel_power_kw: float = Field(
        description="Сумарна встановлена потужність панелей у кВт, більше 0."
    )
    peak_sun_hours: float = Field(
        description="Кількість ефективних сонячних годин на добу, від 0 до 12."
    )
    system_efficiency_percent: float = Field(
        default=80,
        description="Загальна ефективність системи, від 50 до 100%."
    )
    days: int = Field(
        default=1,
        description="Кількість днів розрахунку, від 1 до 365."
    )

    @field_validator("panel_power_kw")
    @classmethod
    def validate_panel_power(cls, value: float) -> float:
        if not 0 < value <= 100_000:
            raise ValueError("Потужність панелей має бути більшою за 0 кВт")
        return value

    @field_validator("peak_sun_hours")
    @classmethod
    def validate_sun_hours(cls, value: float) -> float:
        if not 0 <= value <= 12:
            raise ValueError("Сонячні години мають бути від 0 до 12")
        return value

    @field_validator("system_efficiency_percent")
    @classmethod
    def validate_efficiency(cls, value: float) -> float:
        if not 50 <= value <= 100:
            raise ValueError("Ефективність має бути від 50 до 100%")
        return value

    @field_validator("days")
    @classmethod
    def validate_days(cls, value: int) -> int:
        if not 1 <= value <= 365:
            raise ValueError("Кількість днів має бути від 1 до 365")
        return value


@tool(args_schema=SolarGenerationInput)
def calculate_solar_generation(
    panel_power_kw: float,
    peak_sun_hours: float,
    system_efficiency_percent: float = 80,
    days: int = 1,
) -> str:
    """Оцінити генерацію сонячної електростанції.

    Використовуйте для запитів про орієнтовний виробіток панелей
    за день або декілька днів з урахуванням системних втрат.
    """
    daily_kwh = panel_power_kw * peak_sun_hours * system_efficiency_percent / 100
    total_kwh = daily_kwh * days

    return (
        f"Добова генерація: {daily_kwh:.3f} кВт·год. "
        f"Період: {days} дн. "
        f"Загальна генерація: {total_kwh:.3f} кВт·год."
    )

## 1.4 🟩 ОБОВ’ЯЗКОВО — tool `check_inverter_load`

In [6]:
class InverterLoadInput(BaseModel):
    """Параметри для перевірки навантаження інвертора."""

    model_config = ConfigDict(str_strip_whitespace=True)

    inverter_power_kw: float = Field(
        description="Номінальна потужність інвертора у кВт, більше 0."
    )
    current_load_watts: float = Field(
        description="Поточне сумарне навантаження у ватах, не менше 0."
    )
    warning_threshold_percent: float = Field(
        default=80,
        description="Поріг попередження, від 50 до 100%."
    )
    surge_power_kw: float | None = Field(
        default=None,
        description="Необов'язкова короткочасна пікова потужність у кВт."
    )

    @field_validator("inverter_power_kw")
    @classmethod
    def validate_inverter_power(cls, value: float) -> float:
        if not 0 < value <= 10_000:
            raise ValueError("Потужність інвертора має бути більшою за 0")
        return value

    @field_validator("current_load_watts")
    @classmethod
    def validate_current_load(cls, value: float) -> float:
        if not 0 <= value <= 10_000_000:
            raise ValueError("Навантаження не може бути від'ємним")
        return value

    @field_validator("warning_threshold_percent")
    @classmethod
    def validate_threshold(cls, value: float) -> float:
        if not 50 <= value <= 100:
            raise ValueError("Поріг попередження має бути від 50 до 100%")
        return value

    @field_validator("surge_power_kw")
    @classmethod
    def validate_surge(cls, value: float | None) -> float | None:
        if value is not None and value <= 0:
            raise ValueError("Пікова потужність має бути більшою за 0")
        return value


@tool(args_schema=InverterLoadInput)
def check_inverter_load(
    inverter_power_kw: float,
    current_load_watts: float,
    warning_threshold_percent: float = 80,
    surge_power_kw: float | None = None,
) -> str:
    """Перевірити завантаження інвертора та ризик перевантаження.

    Використовуйте, коли потрібно визначити відсоток завантаження,
    залишковий запас потужності або перевищення номіналу.
    """
    nominal_watts = inverter_power_kw * 1000
    load_percent = current_load_watts / nominal_watts * 100
    reserve_watts = nominal_watts - current_load_watts

    if current_load_watts == 0:
        status = "без навантаження"
    elif load_percent < warning_threshold_percent:
        status = "нормальне навантаження"
    elif load_percent <= 100:
        status = "високе навантаження"
    else:
        status = "перевантаження"

    surge_info = ""
    if surge_power_kw is not None:
        surge_watts = surge_power_kw * 1000
        if current_load_watts > surge_watts:
            surge_info = " Навантаження перевищує також пікову потужність."
        elif current_load_watts > nominal_watts:
            surge_info = " Навантаження вище номіналу, але в межах пікової потужності."

    return (
        f"Завантаження інвертора: {load_percent:.1f}%. "
        f"Запас номінальної потужності: {reserve_watts:.2f} Вт. "
        f"Статус: {status}.{surge_info}"
    )

## 1.5 🟩 ОБОВ’ЯЗКОВО — список tools

In [7]:
tools = [
    calculate_energy,
    estimate_battery_runtime,
    calculate_solar_generation,
    check_inverter_load,
]

tool_map = {current_tool.name: current_tool for current_tool in tools}

print(f"Кількість tools: {len(tools)}")
for current_tool in tools:
    print("-", current_tool.name)

Кількість tools: 4
- calculate_energy
- estimate_battery_runtime
- calculate_solar_generation
- check_inverter_load



## 1.6 🟨 ДЕМОНСТРАЦІЯ — локальні unit tests

Ця комірка **не використовує Gemini API**. Її можна запускати безкоштовно.

Вона демонструє:

- правильні обчислення;
- відхилення некоректних параметрів;
- роботу `field_validator`.


In [8]:
def run_local_tool_tests():
    # calculate_energy
    result = calculate_energy.invoke({
        "power_watts": 1000,
        "hours": 2,
        "devices_count": 2,
    })
    assert "4.000 кВт·год" in result

    # estimate_battery_runtime
    result = estimate_battery_runtime.invoke({
        "battery_capacity_kwh": 10,
        "current_soc_percent": 100,
        "reserve_soc_percent": 10,
        "load_watts": 1000,
        "efficiency_percent": 100,
    })
    assert "9 год 0 хв" in result

    # calculate_solar_generation
    result = calculate_solar_generation.invoke({
        "panel_power_kw": 10,
        "peak_sun_hours": 5,
        "system_efficiency_percent": 80,
        "days": 2,
    })
    assert "80.000 кВт·год" in result

    # check_inverter_load
    result = check_inverter_load.invoke({
        "inverter_power_kw": 10,
        "current_load_watts": 9000,
        "warning_threshold_percent": 80,
    })
    assert "90.0%" in result
    assert "високе навантаження" in result

    invalid_cases = [
        lambda: calculate_energy.invoke({
            "power_watts": -1, "hours": 2, "devices_count": 1
        }),
        lambda: estimate_battery_runtime.invoke({
            "battery_capacity_kwh": 10,
            "current_soc_percent": 101,
            "reserve_soc_percent": 10,
            "load_watts": 1000,
            "efficiency_percent": 90,
        }),
        lambda: calculate_solar_generation.invoke({
            "panel_power_kw": 8,
            "peak_sun_hours": 15,
            "system_efficiency_percent": 80,
            "days": 1,
        }),
        lambda: check_inverter_load.invoke({
            "inverter_power_kw": 6,
            "current_load_watts": -100,
            "warning_threshold_percent": 80,
        }),
    ]

    rejected = 0
    for invalid_call in invalid_cases:
        try:
            invalid_call()
        except Exception:
            rejected += 1

    assert rejected == len(invalid_cases)

    print("✅ Усі локальні unit tests успішно пройдено")
    print(f"✅ Некоректних входів відхилено: {rejected}/{len(invalid_cases)}")


run_local_tool_tests()

✅ Усі локальні unit tests успішно пройдено
✅ Некоректних входів відхилено: 4/4



# 2. Підключення Google Gemini

## Перед запуском

1. У Google Colab відкрийте панель **Secrets**.
2. Створіть секрет `GOOGLE_API_KEY`.
3. Вставте API-ключ.
4. Увімкніть **Notebook access**.

> 🟥 Наступні комірки звертаються до Gemini. Після успішного результату не запускайте їх без потреби.


## 2.1 🟩 ОБОВ’ЯЗКОВО — завантаження API-ключа

In [9]:
try:
    from google.colab import userdata
    google_api_key = userdata.get("GOOGLE_API_KEY")
except ImportError:
    google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError(
        "Не знайдено GOOGLE_API_KEY. Додайте його в Colab Secrets."
    )

os.environ["GOOGLE_API_KEY"] = google_api_key
print("✅ GOOGLE_API_KEY завантажено без виведення ключа")

✅ GOOGLE_API_KEY завантажено без виведення ключа



## 2.2 🟩 ОБОВ’ЯЗКОВО — створення LLM

За потреби змініть лише значення `GEMINI_MODEL` на модель, доступну у вашому API-проєкті.

Не додаємо окремий formatter-LLM, щоб не витрачати додаткову квоту.


In [28]:
GEMINI_MODEL = "gemini-3.6-flash"

# Для деяких моделей temperature ігнорується, тому параметр не передаємо.
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    max_retries=2,
)

llm_with_tools = llm.bind_tools(tools)

print("✅ LLM створено:", GEMINI_MODEL)
print("✅ Tools прив'язано:", len(tools))

✅ LLM створено: gemini-3.6-flash
✅ Tools прив'язано: 4



## 2.3 🟨 ДЕМОНСТРАЦІЯ — один короткий тест підключення

🟥 **Запустити лише один раз.**  
Після отримання відповіді більше не повторювати.


In [11]:
connection_test = llm.invoke(
    "Відповідай українською одним коротким реченням: що таке сонячний інвертор?"
)
print(connection_test.content)

[{'type': 'text', 'text': 'Сонячний інвертор — це пристрій, який перетворює постійний струм від сонячних панелей на змінний струм для живлення побутових приладів.', 'extras': {'signature': 'EsMOCsAOARFNMg/PCPUDsMW/Rvz/DpB3U8lEap9L/JQ3dQp6Vl8/gdjcz9s9n5+/UjoLdz/bI4ulOl4ZAxAweNZ//mZ2Z8krzQ6j8DZGgLPhtQfMwbCLn0uLdJ3B6OIjZnsBWeJRYbNQihfxPj9Kf+og+EbLoTwvC5qZYHgV27Jkx/+OxKmCaDA60ZzFeevnTIMslkUrLtFbdLHmg3Gkm45mjvSKsKP5Uqcyf33Ffe5cmTReAfAfnofCE4y4j52Cl/qQiOXeehH39vmYXSKzbkJkP/NJ5nbiGdG65n5NKLv6mZPP+f1x+DBGotBhyxvwKCvaxBAoo0uiTMyBmVnTExecs1wI1qy0hSCiCw1OES1YyZDb/H0AiMjlFxCgM9vybWkTzbaW5pFrL7so3iPjwPDTnuvhO892pnPW0QQs5gcUsRGhnp0rS3mqfM+NQCdGSI5Uxh/eL4YQRlWP9rQZc8tVtpjLiueHZx2GpVOrHC5CEaDWCZlgVzaeYWNNMSeHUMhjDe+CyLdDtEGx/8Joa8OAVvsVBe8Jgm+Y0EVEKI0ytjp1arkKB7GZK1F3HF59iRhlU30CXWpy/vT8bZUbplez335aumjsW/P6txH2b3cKYu17liV6YuEUWEI2JuLOEWBjbTcMlUKc/7H9NerF2pQl1J3r98tUK1Ymo1tYTL9B5wSTw5TLYtS8OsRAaVnvyJtptdn1+BWjvYM+glhoFSkKFIXiCqJIp+e57+BjM/NAxkxu+ppA6zj4qDPHIJlYhP2SKFlUw+RkcK/2OigKfVmksZ/uRN0Ay1+iApqvq1


# 3. Завдання 2 — LangGraph ReAct-цикл

Архітектура:

```text
START → agent → router
                 ├─ tools → agent
                 └─ END
```

- `agent` викликає Gemini з прив’язаними tools;
- `ToolNode` виконує tool calls;
- router перевіряє, чи є нові tool calls;
- цикл завершується, коли модель формує звичайну відповідь.


## 3.1 🟩 ОБОВ’ЯЗКОВО — стан агента

In [12]:
class AgentState(TypedDict, total=False):
    """Стан, що передається між вузлами LangGraph."""

    messages: Annotated[list, operator.add]
    step_count: int
    started_at: float
    stop_reason: str | None
    recent_tool_signatures: list[str]


MAX_STEPS = 10
TIMEOUT_SECONDS = 90
MAX_REPEATS = 3

## 3.2 🟩 ОБОВ’ЯЗКОВО — системний prompt

In [13]:
SYSTEM_PROMPT = """
Ти — ReAct-агент для аналізу домашньої енергосистеми.

Доступні інструменти:
- calculate_energy — розрахунок споживання;
- estimate_battery_runtime — час роботи акумулятора;
- calculate_solar_generation — оцінка сонячної генерації;
- check_inverter_load — перевірка навантаження інвертора.

Правила:
1. Відповідай українською мовою.
2. Для числових енергетичних розрахунків використовуй відповідний tool.
3. Не вигадуй числові результати.
4. Після tool call поясни результат користувачу.
5. Якщо даних недостатньо, прямо вкажи, яких параметрів бракує.
6. Не повторюй однаковий tool call без причини.
"""

## 3.3 🟩 ОБОВ’ЯЗКОВО — допоміжні функції та logger

In [14]:
def extract_text_content(content) -> str:
    """Перетворити content LangChain/Gemini на звичайний текст."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            else:
                parts.append(str(block))
        return "\n".join(part for part in parts if part)

    return str(content)


def normalize_tool_signature(tool_call: dict) -> str:
    """Стабільний підпис tool call для loop detection."""
    name = tool_call.get("name", "unknown")
    args = tool_call.get("args", {})
    raw = json.dumps(
        {"name": name, "args": args},
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


class TrajectoryLogger:
    """Структуроване логування кроків агента."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.steps = []
        self.started_at = time.monotonic()

    def log(
        self,
        node: str,
        input_data,
        output_data,
        tool_calls=None,
        duration_ms: int = 0,
    ):
        self.steps.append({
            "step_number": len(self.steps) + 1,
            "node_name": node,
            "input": str(input_data)[:500],
            "output": str(output_data)[:500],
            "tool_calls": tool_calls or [],
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "duration_ms": int(duration_ms),
            "elapsed_ms": int(
                (time.monotonic() - self.started_at) * 1000
            ),
        })

    def snapshot(self) -> dict:
        return {
            "total_steps": len(self.steps),
            "total_time_ms": int(
                (time.monotonic() - self.started_at) * 1000
            ),
            "trajectory": self.steps,
        }

    def save(self, filepath: str = "trajectory.json"):
        with open(filepath, "w", encoding="utf-8") as file:
            json.dump(
                self.snapshot(),
                file,
                ensure_ascii=False,
                indent=2,
            )
        print(f"✅ Trajectory saved: {filepath}")


trajectory_logger = TrajectoryLogger()

## 3.4 🟩 ОБОВ’ЯЗКОВО — вузол `agent` із захистами

In [15]:
def agent_node(state: AgentState) -> dict:
    """LLM вирішує: викликати tool чи сформувати фінальну відповідь."""
    node_started = time.monotonic()

    step_count = state.get("step_count", 0)
    started_at = state.get("started_at", node_started)
    elapsed = time.monotonic() - started_at

    # max_steps
    if step_count >= MAX_STEPS:
        message = AIMessage(
            content=(
                f"Досягнуто ліміт кроків ({MAX_STEPS}). "
                "Виконання зупинено з частковим результатом."
            )
        )
        trajectory_logger.log(
            "agent",
            "max_steps check",
            message.content,
            duration_ms=(time.monotonic() - node_started) * 1000,
        )
        return {
            "messages": [message],
            "step_count": step_count,
            "stop_reason": "max_steps",
        }

    # timeout check до виклику LLM
    if elapsed >= TIMEOUT_SECONDS:
        message = AIMessage(
            content=(
                f"Досягнуто загальний timeout ({TIMEOUT_SECONDS} с). "
                "Виконання зупинено."
            )
        )
        trajectory_logger.log(
            "agent",
            "timeout check",
            message.content,
            duration_ms=(time.monotonic() - node_started) * 1000,
        )
        return {
            "messages": [message],
            "step_count": step_count,
            "stop_reason": "timeout",
        }

    llm_messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"],
    ]

    response = llm_with_tools.invoke(llm_messages)

    tool_calls = getattr(response, "tool_calls", []) or []
    signatures = list(state.get("recent_tool_signatures", []))
    signatures.extend(
        normalize_tool_signature(call)
        for call in tool_calls
    )

    # loop detection: три однакові послідовні виклики
    loop_detected = (
        len(signatures) >= MAX_REPEATS
        and len(set(signatures[-MAX_REPEATS:])) == 1
    )

    duration_ms = (time.monotonic() - node_started) * 1000

    if loop_detected:
        stop_message = AIMessage(
            content=(
                f"Виявлено зациклення: той самий tool call повторився "
                f"{MAX_REPEATS} рази поспіль. Виконання зупинено."
            )
        )
        trajectory_logger.log(
            "agent",
            state["messages"][-1],
            stop_message.content,
            tool_calls=tool_calls,
            duration_ms=duration_ms,
        )
        return {
            "messages": [stop_message],
            "step_count": step_count + 1,
            "stop_reason": "loop_detected",
            "recent_tool_signatures": signatures,
        }

    trajectory_logger.log(
        "agent",
        state["messages"][-1],
        extract_text_content(response.content),
        tool_calls=tool_calls,
        duration_ms=duration_ms,
    )

    return {
        "messages": [response],
        "step_count": step_count + 1,
        "started_at": started_at,
        "stop_reason": None,
        "recent_tool_signatures": signatures,
    }

## 3.5 🟩 ОБОВ’ЯЗКОВО — вузол `tools` через `ToolNode`

In [16]:
base_tool_node = ToolNode(tools)


def tools_node(state: AgentState) -> dict:
    """Виконати tool calls через стандартний LangGraph ToolNode."""
    node_started = time.monotonic()

    result = base_tool_node.invoke(state)
    produced_messages = result.get("messages", [])

    trajectory_logger.log(
        "tools",
        state["messages"][-1],
        [
            extract_text_content(message.content)
            for message in produced_messages
        ],
        duration_ms=(time.monotonic() - node_started) * 1000,
    )

    return result

## 3.6 🟩 ОБОВ’ЯЗКОВО — router і компіляція графа

In [29]:
def should_continue(
    state: AgentState,
) -> Literal["tools", "__end__"]:
    """Перейти до tools, якщо LLM повернув tool calls."""
    if state.get("stop_reason"):
        return "__end__"

    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None)

    if tool_calls:
        return "tools"

    return "__end__"


graph_builder = StateGraph(AgentState)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tools_node)

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "__end__": END,
    },
)
graph_builder.add_edge("tools", "agent")

app = graph_builder.compile()

print("✅ LangGraph скомпільовано")
print("START → agent → tools → agent / END")

✅ LangGraph скомпільовано
START → agent → tools → agent / END



# 4. Structured output

За умовою structured output можна реалізувати:

- через `with_structured_output`, **або**
- через Pydantic response model і фінальну обробку відповіді.

Тут використовується другий варіант, тому **додатковий API-виклик не потрібен**.


## 4.1 🟩 ОБОВ’ЯЗКОВО — Pydantic response model

In [18]:
class StructuredAgentResponse(BaseModel):
    """Структурована фінальна відповідь агента."""

    answer: str = Field(
        description="Фінальна відповідь українською мовою."
    )
    confidence: float = Field(
        description="Оцінка впевненості від 0.0 до 1.0."
    )
    sources: list[str] = Field(
        default_factory=list,
        description="Назви tools, використаних під час виконання."
    )

    @field_validator("answer")
    @classmethod
    def validate_answer(cls, value: str) -> str:
        value = value.strip()
        if len(value) < 5:
            raise ValueError("Відповідь має містити мінімум 5 символів")
        return value

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float) -> float:
        if not 0.0 <= value <= 1.0:
            raise ValueError("Confidence має бути від 0.0 до 1.0")
        return value

    @field_validator("sources")
    @classmethod
    def unique_sources(cls, value: list[str]) -> list[str]:
        return list(dict.fromkeys(value))


def collect_tool_names(messages: list) -> list[str]:
    """Зібрати унікальні назви фактично викликаних tools."""
    names = []

    for message in messages:
        calls = getattr(message, "tool_calls", None) or []
        for call in calls:
            name = call.get("name")
            if name and name not in names:
                names.append(name)

    return names


def build_structured_response(
    graph_result: dict,
) -> StructuredAgentResponse:
    """Локально сформувати structured output без другого formatter-LLM."""
    final_text = extract_text_content(
        graph_result["messages"][-1].content
    )
    sources = collect_tool_names(graph_result["messages"])

    confidence = 0.95 if sources else 0.75

    return StructuredAgentResponse(
        answer=final_text,
        confidence=confidence,
        sources=sources,
    )


# 5. Завдання 3 — запуск із max_steps, timeout і loop detection

Функція нижче:

- запускає граф;
- застосовує зовнішній timeout;
- обробляє помилку квоти;
- формує structured output;
- повертає кроки, tools, час і причину зупинки;
- зберігає траєкторію.


## 5.1 🟩 ОБОВ’ЯЗКОВО — безпечний runner

In [19]:
def run_agent(
    query: str,
    timeout_seconds: int = TIMEOUT_SECONDS,
    save_trajectory: bool = True,
) -> dict:
    """Запустити ReAct-агента з timeout і логуванням."""
    if not query or len(query.strip()) < 3:
        raise ValueError("Запит має містити мінімум 3 символи")

    trajectory_logger.reset()
    run_started = time.monotonic()

    initial_state: AgentState = {
        "messages": [HumanMessage(content=query.strip())],
        "step_count": 0,
        "started_at": run_started,
        "stop_reason": None,
        "recent_tool_signatures": [],
    }

    try:
        # Зовнішній timeout на весь app.invoke
        with ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(
                app.invoke,
                initial_state,
                {"recursion_limit": MAX_STEPS * 3},
            )
            graph_result = future.result(timeout=timeout_seconds)

        structured_response = build_structured_response(graph_result)
        elapsed_ms = int((time.monotonic() - run_started) * 1000)

        result = {
            "status": "success",
            "query": query,
            "actual": structured_response.answer,
            "structured_response": structured_response.model_dump(),
            "steps": graph_result.get("step_count", 0),
            "tool_calls": collect_tool_names(
                graph_result.get("messages", [])
            ),
            "elapsed_ms": elapsed_ms,
            "stop_reason": graph_result.get("stop_reason"),
            "graph_result": graph_result,
            "trajectory": trajectory_logger.snapshot(),
        }

    except FuturesTimeoutError:
        result = {
            "status": "timeout",
            "query": query,
            "actual": (
                f"Виконання перевищило timeout {timeout_seconds} секунд."
            ),
            "structured_response": None,
            "steps": len(trajectory_logger.steps),
            "tool_calls": [],
            "elapsed_ms": int(
                (time.monotonic() - run_started) * 1000
            ),
            "stop_reason": "timeout",
            "graph_result": None,
            "trajectory": trajectory_logger.snapshot(),
        }

    except Exception as error:
        error_text = str(error)

        if (
            "429" in error_text
            or "RESOURCE_EXHAUSTED" in error_text
            or "Quota exceeded" in error_text
        ):
            status = "quota_exceeded"
            stop_reason = "quota_exceeded"
        else:
            status = "error"
            stop_reason = "exception"

        result = {
            "status": status,
            "query": query,
            "actual": f"Error: {error_text[:500]}",
            "structured_response": None,
            "steps": len(trajectory_logger.steps),
            "tool_calls": [],
            "elapsed_ms": int(
                (time.monotonic() - run_started) * 1000
            ),
            "stop_reason": stop_reason,
            "graph_result": None,
            "trajectory": trajectory_logger.snapshot(),
        }

    if save_trajectory:
        with open("trajectory.json", "w", encoding="utf-8") as file:
            json.dump(
                result["trajectory"],
                file,
                ensure_ascii=False,
                indent=2,
            )
        print("✅ trajectory.json збережено")

    return result


## 5.2 🟨 ДЕМОНСТРАЦІЯ max_steps, timeout і loop detection без Gemini

Ці демонстрації **не витрачають API-квоту**.  
Вони показують саму логіку захисних механізмів окремо.


In [20]:
def demonstrate_safety_mechanisms():
    print("1. MAX_STEPS")
    simulated_steps = MAX_STEPS
    if simulated_steps >= MAX_STEPS:
        print(
            f"✅ Досягнуто ліміт {MAX_STEPS} кроків — "
            "виконання має бути зупинено."
        )

    print("\n2. TIMEOUT")
    started = time.monotonic() - (TIMEOUT_SECONDS + 1)
    if time.monotonic() - started >= TIMEOUT_SECONDS:
        print(
            f"✅ Перевищено timeout {TIMEOUT_SECONDS} с — "
            "виконання має бути зупинено."
        )

    print("\n3. LOOP DETECTION")
    fake_call = {
        "name": "calculate_energy",
        "args": {
            "power_watts": 1000,
            "hours": 2,
            "devices_count": 1,
        },
    }
    signatures = [
        normalize_tool_signature(fake_call)
        for _ in range(MAX_REPEATS)
    ]
    loop_detected = (
        len(signatures) >= MAX_REPEATS
        and len(set(signatures[-MAX_REPEATS:])) == 1
    )
    assert loop_detected is True
    print(
        f"✅ Однаковий tool call повторився {MAX_REPEATS} рази — "
        "зациклення виявлено."
    )


demonstrate_safety_mechanisms()

1. MAX_STEPS
✅ Досягнуто ліміт 10 кроків — виконання має бути зупинено.

2. TIMEOUT
✅ Перевищено timeout 90 с — виконання має бути зупинено.

3. LOOP DETECTION
✅ Однаковий tool call повторився 3 рази — зациклення виявлено.



# 6. 🟨 ДЕМОНСТРАЦІЯ одного повного ReAct-запуску

🟥 **Ця комірка витрачає Gemini API-квоту. Запустіть її один раз.**

Очікувана траєкторія:

1. `agent` обирає `check_inverter_load`;
2. `tools` виконує обчислення;
3. `agent` формує фінальну відповідь.


In [21]:
demo_result = run_agent(
    "Перевір інвертор потужністю 6 кВт, "
    "якщо поточне навантаження становить 5400 Вт, "
    "а поріг попередження — 80%.",
    save_trajectory=True,
)

print("Статус:", demo_result["status"])
print("Кроків:", demo_result["steps"])
print("Tools:", demo_result["tool_calls"])
print("Час, мс:", demo_result["elapsed_ms"])
print("Відповідь:", demo_result["actual"])

if demo_result["structured_response"]:
    print("\nStructured output:")
    print(json.dumps(
        demo_result["structured_response"],
        ensure_ascii=False,
        indent=2,
    ))

✅ trajectory.json збережено
Статус: success
Кроків: 2
Tools: ['check_inverter_load']
Час, мс: 3352
Відповідь: За результатами перевірки:

- **Номінальна потужність інвертора:** 6 кВт (6000 Вт)
- **Поточне навантаження:** 5400 Вт
- **Завантаження інвертора:** **90%**
- **Запас потужності:** 600 Вт
- **Статус:** **Високе навантаження**

Поточне завантаження (90%) перевищує заданий поріг попередження у 80%. Інвертор працює близько до своєї максимальної номінальної потужності, тому рекомендується зменшити навантаження або утриматися від підключення додаткових потужних приладів, щоб уникнути перегріву чи спрацьовування захисту.

Structured output:
{
  "answer": "За результатами перевірки:\n\n- **Номінальна потужність інвертора:** 6 кВт (6000 Вт)\n- **Поточне навантаження:** 5400 Вт\n- **Завантаження інвертора:** **90%**\n- **Запас потужності:** 600 Вт\n- **Статус:** **Високе навантаження**\n\nПоточне завантаження (90%) перевищує заданий поріг попередження у 80%. Інвертор працює близько до с


# 7. Завдання 4 — фінальні тест-кейси

## Важливо

🟥 Комірку з тестами запускайте **лише один раз**, коли:

- API-ключ уже активний;
- billing / квота достатні;
- усі попередні комірки виконані без помилок.

Використано 4 тест-кейси:

- simple — один tool;
- simple — інший tool;
- medium — два різні обчислення;
- complex — кілька аспектів домашньої енергосистеми.


## 7.1 🟩 ОБОВ’ЯЗКОВО — опис тест-кейсів

In [22]:
test_cases = [
    {
        "id": "TC-001",
        "complexity": "simple",
        "query": (
            "Скільки електроенергії споживає обігрівач "
            "потужністю 2000 Вт за 5 годин?"
        ),
        "expected": (
            "Виклик calculate_energy і результат близько 10 кВт·год."
        ),
    },
    {
        "id": "TC-002",
        "complexity": "simple",
        "query": (
            "Акумулятор має ємність 12 кВт·год, заряд 80%, "
            "резерв 10%, навантаження 1000 Вт і ККД 90%. "
            "На скільки часу його вистачить?"
        ),
        "expected": (
            "Виклик estimate_battery_runtime і результат близько "
            "7 год 34 хв."
        ),
    },
    {
        "id": "TC-003",
        "complexity": "medium",
        "query": (
            "Сонячна станція 8 кВт має 5 ефективних сонячних годин "
            "і ККД 80%. Будинок споживає 1500 Вт протягом 10 годин. "
            "Розрахуй генерацію та споживання і скажи, чи вистачить енергії."
        ),
        "expected": (
            "Виклики calculate_solar_generation і calculate_energy; "
            "генерація 32 кВт·год, споживання 15 кВт·год."
        ),
    },
    {
        "id": "TC-004",
        "complexity": "complex",
        "query": (
            "Є інвертор 6 кВт з навантаженням 5400 Вт, "
            "акумулятор 12 кВт·год із зарядом 75%, резервом 10%, "
            "ККД 92%, а навантаження будинку 2000 Вт. "
            "Оціни завантаження інвертора і час роботи акумулятора."
        ),
        "expected": (
            "Виклики check_inverter_load і estimate_battery_runtime; "
            "завантаження інвертора 90% та автономність близько 3 год 35 хв."
        ),
    },
]

print(f"Підготовлено тест-кейсів: {len(test_cases)}")

Підготовлено тест-кейсів: 4


## 7.2 🟥 НЕ ЗАПУСКАТИ ПОВТОРНО — фінальний test runner

In [30]:
test_results = []
all_trajectories = []

for test_case in test_cases:
    print("\n" + "=" * 80)
    print(test_case["id"], "-", test_case["complexity"])
    print(test_case["query"])

    run_result = run_agent(
        test_case["query"],
        save_trajectory=False,
    )

    test_results.append({
        "test_id": test_case["id"],
        "query": test_case["query"],
        "expected": test_case["expected"],
        "actual": run_result["actual"][:1000],
        "status": run_result["status"],
        "steps": run_result["steps"],
        "tool_calls": run_result["tool_calls"],
        "elapsed_ms": run_result["elapsed_ms"],
        "complexity": test_case["complexity"],
        "stop_reason": run_result["stop_reason"],
        "max_steps_reached": (
            run_result["stop_reason"] == "max_steps"
        ),
        "timeout_reached": (
            run_result["stop_reason"] == "timeout"
        ),
    })

    all_trajectories.append({
        "test_id": test_case["id"],
        **run_result["trajectory"],
    })

    print("Статус:", run_result["status"])
    print("Кроків:", run_result["steps"])
    print("Tools:", run_result["tool_calls"])
    print("Час, мс:", run_result["elapsed_ms"])


with open("test_results.json", "w", encoding="utf-8") as file:
    json.dump(
        test_results,
        file,
        ensure_ascii=False,
        indent=2,
    )

with open("trajectory.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "runs": all_trajectories,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print("\n✅ test_results.json збережено")
print("✅ trajectory.json збережено")


TC-001 - simple
Скільки електроенергії споживає обігрівач потужністю 2000 Вт за 5 годин?
Статус: success
Кроків: 2
Tools: ['calculate_energy']
Час, мс: 2504

TC-002 - simple
Акумулятор має ємність 12 кВт·год, заряд 80%, резерв 10%, навантаження 1000 Вт і ККД 90%. На скільки часу його вистачить?
Статус: success
Кроків: 2
Tools: ['estimate_battery_runtime']
Час, мс: 3531

TC-003 - medium
Сонячна станція 8 кВт має 5 ефективних сонячних годин і ККД 80%. Будинок споживає 1500 Вт протягом 10 годин. Розрахуй генерацію та споживання і скажи, чи вистачить енергії.
Статус: quota_exceeded
Кроків: 2
Tools: []
Час, мс: 5727

TC-004 - complex
Є інвертор 6 кВт з навантаженням 5400 Вт, акумулятор 12 кВт·год із зарядом 75%, резервом 10%, ККД 92%, а навантаження будинку 2000 Вт. Оціни завантаження інвертора і час роботи акумулятора.
Статус: quota_exceeded
Кроків: 0
Tools: []
Час, мс: 1725

✅ test_results.json збережено
✅ trajectory.json збережено


In [35]:
# Повторний запуск ТІЛЬКИ тестів, які завершилися помилкою.
# Успішні TC-001 і TC-002 повторно не запускаються.

failed_statuses = {
    "quota_exceeded",
    "error",
    "timeout",
}

failed_ids = {
    result["test_id"]
    for result in test_results
    if result["status"] in failed_statuses
}

print("Повторно запускаємо тільки:", sorted(failed_ids))

updated_results = []
new_trajectories = []

for old_result in test_results:
    test_id = old_result["test_id"]

    # Залишаємо успішний результат без повторного API-виклику
    if test_id not in failed_ids:
        updated_results.append(old_result)
        print(f"✅ {test_id} залишено без повторного запуску")
        continue

    test_case = next(
        tc for tc in test_cases
        if tc["id"] == test_id
    )

    print("\n" + "=" * 80)
    print(f"Повторний запуск {test_id}")
    print(test_case["query"])

    run_result = run_agent(
        test_case["query"],
        save_trajectory=False,
    )

    updated_results.append({
        "test_id": test_case["id"],
        "query": test_case["query"],
        "expected": test_case["expected"],
        "actual": run_result["actual"][:1000],
        "status": run_result["status"],
        "steps": run_result["steps"],
        "tool_calls": run_result["tool_calls"],
        "elapsed_ms": run_result["elapsed_ms"],
        "complexity": test_case["complexity"],
        "stop_reason": run_result["stop_reason"],
        "max_steps_reached": (
            run_result["stop_reason"] == "max_steps"
        ),
        "timeout_reached": (
            run_result["stop_reason"] == "timeout"
        ),
    })

    new_trajectories.append({
        "test_id": test_case["id"],
        **run_result["trajectory"],
    })

    print("Статус:", run_result["status"])
    print("Кроків:", run_result["steps"])
    print("Tools:", run_result["tool_calls"])


# Замінюємо старий список оновленим
test_results = updated_results

# Перезаписуємо фінальні результати
with open("test_results.json", "w", encoding="utf-8") as file:
    json.dump(
        test_results,
        file,
        ensure_ascii=False,
        indent=2,
    )

# Додаємо нові успішні траєкторії до trajectory.json
try:
    with open("trajectory.json", "r", encoding="utf-8") as file:
        trajectory_data = json.load(file)
except Exception:
    trajectory_data = {"runs": []}

existing_runs = trajectory_data.get("runs", [])

rerun_ids = {
    item["test_id"]
    for item in new_trajectories
}

# Прибираємо старі невдалі траєкторії TC-003/TC-004
existing_runs = [
    item for item in existing_runs
    if item.get("test_id") not in rerun_ids
]

trajectory_data = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "runs": existing_runs + new_trajectories,
}

with open("trajectory.json", "w", encoding="utf-8") as file:
    json.dump(
        trajectory_data,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("\n✅ test_results.json оновлено")
print("✅ trajectory.json оновлено")

Повторно запускаємо тільки: ['TC-003', 'TC-004']
✅ TC-001 залишено без повторного запуску
✅ TC-002 залишено без повторного запуску

Повторний запуск TC-003
Сонячна станція 8 кВт має 5 ефективних сонячних годин і ККД 80%. Будинок споживає 1500 Вт протягом 10 годин. Розрахуй генерацію та споживання і скажи, чи вистачить енергії.
Статус: success
Кроків: 2
Tools: ['calculate_solar_generation', 'calculate_energy']

Повторний запуск TC-004
Є інвертор 6 кВт з навантаженням 5400 Вт, акумулятор 12 кВт·год із зарядом 75%, резервом 10%, ККД 92%, а навантаження будинку 2000 Вт. Оціни завантаження інвертора і час роботи акумулятора.
Статус: success
Кроків: 2
Tools: ['check_inverter_load', 'estimate_battery_runtime']

✅ test_results.json оновлено
✅ trajectory.json оновлено


## 7.3 🟩 ОБОВ’ЯЗКОВО — таблиця результатів

In [36]:
import pandas as pd

results_table = pd.DataFrame([
    {
        "Test ID": item["test_id"],
        "Складність": item["complexity"],
        "Статус": item["status"],
        "Кроків": item["steps"],
        "Tools": ", ".join(item["tool_calls"]),
        "Час, мс": item["elapsed_ms"],
    }
    for item in test_results
])

display(results_table)

,Test ID,Складність,Статус,Кроків,Tools,"Час, мс"
0,TC-001,simple,success,2,calculate_energy,2504
1,TC-002,simple,success,2,estimate_battery_runtime,3531
2,TC-003,medium,success,2,"calculate_solar_generation, calculate_energy",5697
3,TC-004,complex,success,2,"check_inverter_load, estimate_battery_runtime",9195



# 8. Автоматичне створення README.md

README містить:

- опис проєкту;
- домен;
- інструменти;
- архітектуру;
- інструкцію запуску;
- результати тестування;
- аналіз;
- обмеження;
- структуру артефактів.


## 8.1 🟩 ОБОВ’ЯЗКОВО — генерація README.md

In [37]:
successful = [
    item for item in test_results
    if item["status"] == "success"
]

avg_time = (
    int(sum(item["elapsed_ms"] for item in successful) / len(successful))
    if successful
    else 0
)

table_lines = [
    "| Test ID | Складність | Кроків | Tool calls | Час, мс | Статус |",
    "|---|---:|---:|---|---:|---|",
]

for item in test_results:
    table_lines.append(
        f"| {item['test_id']} | {item['complexity']} | "
        f"{item['steps']} | {', '.join(item['tool_calls']) or '—'} | "
        f"{item['elapsed_ms']} | {item['status']} |"
    )

results_markdown = "\n".join(table_lines)

readme_text = f"""# ReAct-агент для аналізу домашньої енергосистеми

## 1. Опис проєкту

У проєкті реалізовано ReAct-агента у LangGraph для виконання
розрахунків домашньої енергосистеми. Агент використовує Google Gemini
через LangChain, самостійно обирає відповідний інструмент, аналізує
результат і формує фінальну відповідь українською мовою.
Рішення оформлено як Google Colab / Jupyter Notebook.

## 2. Доменна задача

- **Домен:** домашня енергетика.
- **Що вирішує агент:** розрахунок споживання, генерації,
  автономності акумулятора та навантаження інвертора.
- **Tools:**
  - `calculate_energy` — споживання електроенергії;
  - `estimate_battery_runtime` — час роботи акумулятора;
  - `calculate_solar_generation` — сонячна генерація;
  - `check_inverter_load` — завантаження інвертора.

## 3. Архітектура

- LLM: `{GEMINI_MODEL}`;
- граф: `START → agent → tools → agent / END`;
- `StateGraph` передає повідомлення і службовий стан;
- `ToolNode` виконує tool calls;
- router завершує граф, коли tool calls відсутні;
- `MAX_STEPS = {MAX_STEPS}`;
- `TIMEOUT_SECONDS = {TIMEOUT_SECONDS}`;
- `MAX_REPEATS = {MAX_REPEATS}`;
- structured output реалізовано через Pydantic response model.

## 4. Інструкція запуску

1. Відкрити файл `.ipynb` у Google Colab.
2. Додати секрет `GOOGLE_API_KEY`.
3. Запустити комірки зверху вниз.
4. Не запускати повторно комірки, позначені червоним.
5. Після тестів завантажити:
   - notebook `.ipynb`;
   - `trajectory.json`;
   - `test_results.json`;
   - `README.md`.

## 5. Результати тестування

{results_markdown}

Успішно виконано: {len(successful)} із {len(test_results)} тестів.
Середній час успішного тесту: {avg_time} мс.

## 6. Аналіз результатів

### Чи правильно агент обирав tools?

У простих запитах агент мав обирати один спеціалізований tool.
У складніших запитах агент мав використати кілька tools, після чого
синтезувати їх результати у спільну відповідь. Фактично використані
інструменти наведені в таблиці та у `test_results.json`.

### Де агент може помилятися?

Основні можливі помилки: недостатні вхідні дані, неточний вибір tool,
повторний виклик з однаковими аргументами або обмеження API.
Pydantic-валідація відхиляє некоректні числові параметри.

### Simple vs complex

Simple-тести зазвичай потребують одного tool call.
Medium і complex можуть містити два tool calls. Кількість agent-кроків
не завжди зростає лінійно, оскільки модель може викликати кілька tools
одночасно.

### Max steps і timeout

Захист `max_steps` обмежує кількість ітерацій.
Зовнішній timeout обмежує загальний час одного запуску.
LoopDetector зупиняє три однакові послідовні tool calls.

### Час виконання

Latency залежить переважно від кількості звернень до LLM.
Локальні Pydantic-перевірки і математичні tools виконуються швидко.

## 7. Висновки та обмеження

**Працює добре:**
- Pydantic-валідація;
- автоматичний вибір tools;
- ReAct-цикл;
- structured output;
- JSON-логування.

**Можна покращити:**
- підключити реальні дані Home Assistant або сонячного API;
- додати retry із backoff;
- розширити набір тестів.

**Обмеження:**
- залежність від доступності та квоти Gemini API;
- результати є інженерними оцінками, а не вимірюваннями;
- timeout через ThreadPoolExecutor не гарантує примусове завершення
  мережевого потоку на всіх платформах.

## 8. Структура артефактів

- `HW1_Babenko_ReAct_Agent_CLEAN.ipynb` — основний notebook;
- `trajectory.json` — траєкторії тестових запусків;
- `test_results.json` — результати тест-кейсів;
- `README.md` — опис, інструкція та аналіз.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_text)

print("✅ README.md збережено")
print(readme_text[:1500], "\n...")

✅ README.md збережено
# ReAct-агент для аналізу домашньої енергосистеми

## 1. Опис проєкту

У проєкті реалізовано ReAct-агента у LangGraph для виконання
розрахунків домашньої енергосистеми. Агент використовує Google Gemini
через LangChain, самостійно обирає відповідний інструмент, аналізує
результат і формує фінальну відповідь українською мовою.
Рішення оформлено як Google Colab / Jupyter Notebook.

## 2. Доменна задача

- **Домен:** домашня енергетика.
- **Що вирішує агент:** розрахунок споживання, генерації,
  автономності акумулятора та навантаження інвертора.
- **Tools:**
  - `calculate_energy` — споживання електроенергії;
  - `estimate_battery_runtime` — час роботи акумулятора;
  - `calculate_solar_generation` — сонячна генерація;
  - `check_inverter_load` — завантаження інвертора.

## 3. Архітектура

- LLM: `gemini-3.6-flash`;
- граф: `START → agent → tools → agent / END`;
- `StateGraph` передає повідомлення і службовий стан;
- `ToolNode` виконує tool calls;
- router завершує гр


# 9. Фінальна перевірка артефактів

Перед здачею повинні існувати:

- `trajectory.json`;
- `test_results.json`;
- `README.md`;
- сам notebook `.ipynb`.


## 9.1 🟩 ОБОВ’ЯЗКОВО — перевірка файлів

In [38]:
from pathlib import Path

required_files = [
    "trajectory.json",
    "test_results.json",
    "README.md",
]

for filename in required_files:
    path = Path(filename)
    if path.exists():
        print(
            f"✅ {filename}: {path.stat().st_size} bytes"
        )
    else:
        print(f"❌ {filename}: файл не знайдено")

✅ trajectory.json: 13607 bytes
✅ test_results.json: 5926 bytes
✅ README.md: 5722 bytes


## 9.2 ⬜ ОПЦІЙНО — завантажити файли з Colab

In [39]:
try:
    from google.colab import files

    for filename in [
        "trajectory.json",
        "test_results.json",
        "README.md",
    ]:
        files.download(filename)

except ImportError:
    print(
        "Ця комірка призначена для Google Colab. "
        "У локальному Jupyter файли вже знаходяться в робочій папці."
    )

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>